# Independent CNN Baseline (Fig. 1 left) — Google Colab

Train **two separate EfficientNet models** (crop + disease), save both checkpoints to Google Drive, then evaluate composition zero-shot performance.

$$\mathrm{score}(c,d \mid x) = \log p_{\mathrm{crop}}(c \mid x) + \log p_{\mathrm{disease}}(d \mid x)$$

**DPD-Small**

| Split | Images |
|-------|--------|
| Train | 8 000 |
| Test Seen | 2 000 |
| Test Unseen | 1 448 |
| **Total** | **11 448** |

**This notebook**
1. Mounts Google Drive  
2. Installs dependencies  
3. Trains crop model → `indep_crop_effb0/best.pt`  
4. Trains disease model → `indep_disease_effb0/best.pt`  
5. Evaluates both together (Top-N, HM, calibration, confusion matrices)  


## 0. Colab setup — Drive mount & installs

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
# Install deps (timm not preinstalled on Colab)
!pip install -q timm tqdm pyyaml scikit-learn


In [3]:
from __future__ import annotations

import csv
import json
import math
import os
import random
import time
from dataclasses import dataclass

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from PIL import Image
from sklearn.metrics import confusion_matrix
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T
from tqdm.auto import tqdm

import timm

print(f"torch {torch.__version__} | cuda={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


torch 2.11.0+cu130 | cuda=True
GPU: Tesla T4


## 1. Paths on Google Drive

**Edit `DRIVE_ROOT`** to where you put the DPD-Small dataset on Drive.

Expected layout under `DRIVE_ROOT`:
```
DRIVE_ROOT/
  dataset/
    DPD_Small_dataset/          # images
    DPD_Small_train.csv
    DPD_Small_test_seen.csv
    DPD_Small_test_unseen.csv
  outputs/                      # created automatically
```


In [8]:
# ========== EDIT THIS ==========
DRIVE_ROOT = "/content/drive/MyDrive/Assignment2COS30082"   # <- change if your folder name differs
# ===============================

DATA_ROOT = DRIVE_ROOT
OUTPUT_DIR = os.path.join(DRIVE_ROOT, "outputs")
IMAGE_DIR_NAME = "DPD_Small_dataset"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("DRIVE_ROOT exists:", os.path.isdir(DRIVE_ROOT))
print("DATA_ROOT exists:", os.path.isdir(DATA_ROOT))
if os.path.isdir(DATA_ROOT):
    print("Contents:", os.listdir(DATA_ROOT)[:20])


DRIVE_ROOT exists: True
DATA_ROOT exists: True
Contents: ['DPD_Small_test_unseen.csv', 'DPD_Small_train.csv', 'DPD_Small_test_seen.csv', 'DPD_Small_dataset', 'Models', 'outputs']


## 2. Configuration (matches baseline.yaml)

In [9]:
CFG = {
    "seed": 42,
    "output_dir": OUTPUT_DIR,
    "data": {
        "root": DATA_ROOT,
        "image_dir": IMAGE_DIR_NAME,
        "train_csv": "DPD_Small_train.csv",
        "test_seen_csv": "DPD_Small_test_seen.csv",
        "test_unseen_csv": "DPD_Small_test_unseen.csv",
        "val_fraction": 0.1,
        "img_size": 224,
        "num_workers": 2,          # Colab: 2 is safer than 4
    },
    "model": {
        "backbone": "efficientnet_b0",
        "pretrained": True,
        "dropout": 0.3,
        "drop_path_rate": 0.2,
    },
    "train": {
        "epochs": 30,
        "batch_size": 64,
        "lr_backbone": 2.0e-4,
        "lr_head": 1.0e-3,
        "weight_decay": 1.0e-2,
        "warmup_epochs": 2,
        "label_smoothing": 0.1,
        "amp": True,
        "patience": 8,
    },
}

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

seed_everything(CFG["seed"])
DEVICE = get_device()
print(f"device = {DEVICE}")
print(f"output_dir = {CFG['output_dir']}")


device = cuda
output_dir = /content/drive/MyDrive/Assignment2COS30082/outputs


## 3. Dataset utilities

In [10]:
COLUMNS = ["image", "crop_label", "disease_label", "comp_label",
           "crop_name", "disease_name", "comp_name"]
_IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff")


def _is_int(x) -> bool:
    try:
        int(str(x).strip())
        return True
    except ValueError:
        return False


def read_split(csv_path: str) -> pd.DataFrame:
    """Read one metadata CSV, with or without a header row."""
    df = pd.read_csv(csv_path)
    if df.shape[1] != len(COLUMNS):
        raise ValueError(f"{csv_path}: expected 7 columns, found {df.shape[1]}")
    if str(df.columns[0]).lower().endswith(_IMG_EXT) or _is_int(df.columns[1]):
        df = pd.read_csv(csv_path, header=None)
    df.columns = COLUMNS
    df["image"] = df["image"].astype(str)
    df["crop_label"] = df["crop_label"].astype(int)
    df["disease_label"] = df["disease_label"].astype(int)
    df["comp_label"] = pd.to_numeric(df["comp_label"], errors="coerce")
    return df.reset_index(drop=True)


def _pairs(df: pd.DataFrame) -> set[tuple[int, int]]:
    return set(zip(df["crop_label"].tolist(), df["disease_label"].tolist()))


@dataclass
class LabelSpace:
    num_crops: int
    num_diseases: int
    seen_pairs: list[tuple[int, int]]
    unseen_pairs: list[tuple[int, int]]
    crop_names: dict[int, str]
    disease_names: dict[int, str]

    @property
    def pairs(self) -> list[tuple[int, int]]:
        return self.seen_pairs + self.unseen_pairs

    @property
    def num_seen(self) -> int:
        return len(self.seen_pairs)

    def pair_index(self) -> dict[tuple[int, int], int]:
        return {p: i for i, p in enumerate(self.pairs)}

    def pair_name(self, i: int) -> str:
        c, d = self.pairs[i]
        return f"{self.crop_names.get(c, c)} | {self.disease_names.get(d, d)}"


def build_label_space(train_df, seen_df, unseen_df) -> LabelSpace:
    all_df = pd.concat([train_df, seen_df, unseen_df], ignore_index=True)
    seen_pairs = sorted(_pairs(train_df))
    unseen_pairs = sorted(_pairs(unseen_df) - set(seen_pairs))
    return LabelSpace(
        num_crops=int(all_df["crop_label"].max()) + 1,
        num_diseases=int(all_df["disease_label"].max()) + 1,
        seen_pairs=seen_pairs,
        unseen_pairs=unseen_pairs,
        crop_names=dict(zip(all_df["crop_label"].tolist(), all_df["crop_name"].astype(str))),
        disease_names=dict(zip(all_df["disease_label"].tolist(), all_df["disease_name"].astype(str))),
    )


def audit(train_df, seen_df, unseen_df, ls: LabelSpace) -> None:
    print(f"[data] train={len(train_df)} test_seen={len(seen_df)} test_unseen={len(unseen_df)}")
    print(f"[data] crops={ls.num_crops} diseases={ls.num_diseases} "
          f"seen pairs={ls.num_seen} unseen pairs={len(ls.unseen_pairs)}")
    extra = _pairs(seen_df) - set(ls.seen_pairs)
    if extra:
        print(f"[warn] {len(extra)} test_seen pairs never in train")
    overlap = _pairs(unseen_df) & set(ls.seen_pairs)
    if overlap:
        print(f"[warn] {len(overlap)} test_unseen pairs also in train")
    tc, td = set(train_df["crop_label"]), set(train_df["disease_label"])
    new_c = set(unseen_df["crop_label"]) - tc
    new_d = set(unseen_df["disease_label"]) - td
    if new_c or new_d:
        print(f"[warn] unseen has new crops {sorted(new_c)} / diseases {sorted(new_d)}")
    else:
        print("[data] compositional zero-shot setting")
    counts = train_df.groupby(["crop_label", "disease_label"]).size()
    print(f"[data] images/seen-pair: min={counts.min()} median={int(counts.median())} max={counts.max()}")


def split_train_val(df: pd.DataFrame, val_fraction: float, seed: int):
    if val_fraction <= 0:
        return df.reset_index(drop=True), df.iloc[0:0].copy()
    rng = np.random.default_rng(seed)
    val_idx: list[int] = []
    for _, g in df.groupby(["crop_label", "disease_label"]):
        idx = g.index.to_numpy().copy()
        rng.shuffle(idx)
        n_val = min(int(len(idx) * val_fraction), len(idx) - 1)
        val_idx.extend(idx[:n_val].tolist())
    is_val = df.index.isin(val_idx)
    return df[~is_val].reset_index(drop=True), df[is_val].reset_index(drop=True)


class PlantDiseaseDataset(Dataset):
    def __init__(self, df: pd.DataFrame, image_dir: str, transform=None):
        self.paths = [os.path.join(image_dir, p) for p in df["image"]]
        self.crops = df["crop_label"].to_numpy()
        self.diseases = df["disease_label"].to_numpy()
        self.transform = transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, i: int) -> dict:
        img = Image.open(self.paths[i]).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return {"image": img, "crop": int(self.crops[i]),
                "disease": int(self.diseases[i]), "index": i}


IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def build_transforms(img_size: int, train: bool):
    if train:
        return T.Compose([
            T.RandomResizedCrop(img_size, scale=(0.6, 1.0)),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(p=0.2),
            T.TrivialAugmentWide(),
            T.ToTensor(),
            T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            T.RandomErasing(p=0.25),
        ])
    return T.Compose([
        T.Resize(int(img_size / 0.875)),
        T.CenterCrop(img_size),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

print("dataset utilities ready")


dataset utilities ready


## 4. Model (single-head EfficientNet per concept)

In [11]:
class EfficientNetHeads(nn.Module):
    """One backbone + linear head(s). Independent models use a single head each."""

    def __init__(self, backbone: str, heads: dict[str, int], pretrained: bool = True,
                 dropout: float = 0.3, drop_path_rate: float = 0.2):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0,
                                          drop_path_rate=drop_path_rate)
        dim = self.backbone.num_features
        self.heads = nn.ModuleDict({
            name: nn.Sequential(nn.Dropout(dropout), nn.Linear(dim, n))
            for name, n in heads.items()
        })

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        feats = self.backbone(x)
        return {name: head(feats) for name, head in self.heads.items()}


def build_model(cfg: dict, heads: dict[str, int], pretrained: bool | None = None) -> EfficientNetHeads:
    m = cfg["model"]
    return EfficientNetHeads(
        backbone=m["backbone"],
        heads=heads,
        pretrained=m.get("pretrained", True) if pretrained is None else pretrained,
        dropout=m.get("dropout", 0.3),
        drop_path_rate=m.get("drop_path_rate", 0.2),
    )


class MultiTaskLoss(nn.Module):
    def __init__(self, weights: dict[str, float], label_smoothing: float = 0.0):
        super().__init__()
        self.weights = weights
        self.ce = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    def forward(self, outputs: dict, targets: dict):
        parts = {k: self.ce(outputs[k], targets[k]) for k in self.weights}
        total = sum(self.weights[k] * v for k, v in parts.items())
        return total, {k: v.detach().item() for k, v in parts.items()}

print("model ready")


model ready


## 5. Train independent models

Each call saves:
- `best.pt` — full checkpoint (weights + heads + config) — **used for eval**
- `best_weights_only.pt` — state_dict only
- `last.pt` — last epoch
- `config.yaml` / `history.csv`


In [12]:
def cosine_with_warmup(optimizer, warmup_steps: int, total_steps: int, min_ratio: float = 0.01):
    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def run_epoch(model, loader, criterion, device, amp, optimizer=None, scheduler=None, scaler=None):
    train = optimizer is not None
    model.train(train)
    heads = list(criterion.weights)
    total_loss, n = 0.0, 0
    correct = {h: 0 for h in heads}
    for batch in tqdm(loader, leave=False, desc="train" if train else "val"):
        x = batch["image"].to(device, non_blocking=True)
        targets = {h: batch[h].to(device, non_blocking=True) for h in heads}
        with torch.set_grad_enabled(train), \
                torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
            out = model(x)
            loss, _ = criterion(out, targets)
        if train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
        bs = x.size(0)
        total_loss += loss.item() * bs
        n += bs
        for h in heads:
            correct[h] += (out[h].argmax(1) == targets[h]).sum().item()
    return {"loss": total_loss / max(n, 1), **{f"{h}_acc": correct[h] / max(n, 1) for h in heads}}


def train_independent(mode: str, cfg: dict = CFG):
    """Train a single-head model. Saves best.pt + last.pt with full config inside."""
    assert mode in ("crop", "disease"), mode
    seed_everything(cfg["seed"])
    device = get_device()
    tc, dc = cfg["train"], cfg["data"]
    amp = bool(tc.get("amp", True)) and device.type == "cuda"
    experiment = f"indep_{mode}_effb0"

    root = dc["root"]
    train_full = read_split(os.path.join(root, dc["train_csv"]))
    seen_df = read_split(os.path.join(root, dc["test_seen_csv"]))
    unseen_df = read_split(os.path.join(root, dc["test_unseen_csv"]))
    ls = build_label_space(train_full, seen_df, unseen_df)
    audit(train_full, seen_df, unseen_df, ls)

    train_df, val_df = split_train_val(train_full, dc["val_fraction"], cfg["seed"])
    print(f"[data] train={len(train_df)} val={len(val_df)}")
    image_dir = os.path.join(root, dc["image_dir"])
    loader_kw = dict(batch_size=tc["batch_size"], num_workers=dc["num_workers"],
                     pin_memory=device.type == "cuda")
    train_loader = DataLoader(
        PlantDiseaseDataset(train_df, image_dir, build_transforms(dc["img_size"], True)),
        shuffle=True, drop_last=len(train_df) > tc["batch_size"], **loader_kw)
    val_loader = None
    if len(val_df):
        val_loader = DataLoader(
            PlantDiseaseDataset(val_df, image_dir, build_transforms(dc["img_size"], False)),
            shuffle=False, **loader_kw)

    num_classes = ls.num_crops if mode == "crop" else ls.num_diseases
    heads = {mode: num_classes}
    model = build_model(cfg, heads).to(device)
    print(f"[model] {cfg['model']['backbone']} heads={heads} "
          f"params={sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

    optimizer = torch.optim.AdamW([
        {"params": model.backbone.parameters(), "lr": tc["lr_backbone"]},
        {"params": model.heads.parameters(), "lr": tc["lr_head"]},
    ], weight_decay=tc["weight_decay"])
    steps_per_epoch = len(train_loader)
    scheduler = cosine_with_warmup(optimizer, tc["warmup_epochs"] * steps_per_epoch,
                                   tc["epochs"] * steps_per_epoch)
    criterion = MultiTaskLoss({mode: 1.0}, tc["label_smoothing"])
    scaler = torch.amp.GradScaler(device.type, enabled=amp)

    out_dir = os.path.join(cfg["output_dir"], experiment)
    os.makedirs(out_dir, exist_ok=True)

    # Config embedded in checkpoint MUST be self-contained for later load
    run_cfg = {
        "seed": cfg["seed"],
        "output_dir": cfg["output_dir"],
        "mode": mode,
        "experiment": experiment,
        "data": dict(cfg["data"]),
        "model": dict(cfg["model"]),
        "train": dict(cfg["train"]),
    }
    with open(os.path.join(out_dir, "config.yaml"), "w") as f:
        yaml.safe_dump(run_cfg, f, sort_keys=False)
    history_path = os.path.join(out_dir, "history.csv")

    best_score, best_epoch, history = -1.0, -1, []
    for epoch in range(1, tc["epochs"] + 1):
        t0 = time.time()
        tr = run_epoch(model, train_loader, criterion, device, amp, optimizer, scheduler, scaler)
        va = run_epoch(model, val_loader, criterion, device, amp) if val_loader else tr
        score = float(va[f"{mode}_acc"])

        row = {"epoch": epoch, **{f"train_{k}": v for k, v in tr.items()},
               **{f"val_{k}": v for k, v in va.items()}, "val_score": score,
               "lr_backbone": optimizer.param_groups[0]["lr"], "time_s": time.time() - t0}
        history.append(row)
        with open(history_path, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(row))
            w.writeheader()
            w.writerows(history)

        ckpt = {
            "model": model.state_dict(),
            "heads": heads,
            "config": run_cfg,
            "epoch": epoch,
            "val": va,
            "num_crops": ls.num_crops,
            "num_diseases": ls.num_diseases,
        }
        torch.save(ckpt, os.path.join(out_dir, "last.pt"))
        improved = score > best_score
        if improved:
            best_score, best_epoch = score, epoch
            torch.save(ckpt, os.path.join(out_dir, "best.pt"))
            torch.save(model.state_dict(), os.path.join(out_dir, "best_weights_only.pt"))
        print(f"epoch {epoch:3d} | train loss {tr['loss']:.4f} | val loss {va['loss']:.4f} "
              f"{mode}={va[f'{mode}_acc']:.4f}{' *' if improved else ''} | {row['time_s']:.0f}s")

        if tc.get("patience", 0) and epoch - best_epoch >= tc["patience"]:
            print(f"[train] early stop after {tc['patience']} epochs without improvement")
            break

    best_path = os.path.join(out_dir, "best.pt")
    print(f"[train] best val {mode} acc {best_score:.4f} @ epoch {best_epoch}")
    print(f"[train] saved: {best_path}")
    print(f"[train] saved: {os.path.join(out_dir, 'best_weights_only.pt')}")
    return best_path, out_dir, ls

print("train_independent() ready")


train_independent() ready


### 5a. Train crop model

In [13]:
CROP_CKPT, CROP_DIR, LS = train_independent("crop")
print("Crop checkpoint:", CROP_CKPT)
print("Files:", os.listdir(CROP_DIR))


[data] train=8000 test_seen=2000 test_unseen=1448
[data] crops=55 diseases=175 seen pairs=323 unseen pairs=10
[data] compositional zero-shot setting
[data] images/seen-pair: min=1 median=7 max=544
[data] train=7329 val=671


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

[model] efficientnet_b0 heads={'crop': 55} params=4.1M


train:   0%|          | 0/114 [00:00<?, ?it/s]

...

epoch  26 | train loss 0.7855 | val loss 0.7933 crop=0.9806 | 209s


train:   0%|          | 0/114 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ceaeb296c00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ceaeb296c00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

val:   0%|          | 0/11 [00:00<?, ?it/s]

epoch  27 | train loss 0.7844 | val loss 0.7924 crop=0.9791 | 223s
[train] early stop after 8 epochs without improvement
[train] best val crop acc 0.9851 @ epoch 19
[train] saved: /content/drive/MyDrive/Assignment2COS30082/outputs/indep_crop_effb0/best.pt
[train] saved: /content/drive/MyDrive/Assignment2COS30082/outputs/indep_crop_effb0/best_weights_only.pt
Crop checkpoint: /content/drive/MyDrive/Assignment2COS30082/outputs/indep_crop_effb0/best.pt
Files: ['config.yaml', 'history.csv', 'last.pt', 'best.pt', 'best_weights_only.pt']


Early stopped at epoch 27 (crop model)

### 5b. Train disease model

In [14]:
DISEASE_CKPT, DISEASE_DIR, LS = train_independent("disease")
print("Disease checkpoint:", DISEASE_CKPT)
print("Files:", os.listdir(DISEASE_DIR))


[data] train=8000 test_seen=2000 test_unseen=1448
[data] crops=55 diseases=175 seen pairs=323 unseen pairs=10
[data] compositional zero-shot setting
[data] images/seen-pair: min=1 median=7 max=544
[data] train=7329 val=671
[model] efficientnet_b0 heads={'disease': 175} params=4.2M


train:   0%|          | 0/114 [00:00<?, ?it/s]

val:   0%|          | 0/11 [00:00<?, ?it/s]

epoch   1 | train loss 4.2635 | val loss 2.7299 disease=0.4754 * | 214s


train:   0%|          | 0/114 [00:00<?, ?it/s]

...

epoch  29 | train loss 1.0391 | val loss 1.2620 disease=0.8867 | 239s


train:   0%|          | 0/114 [00:00<?, ?it/s]

Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ceaeb296c00><function _MultiProcessingDataLoaderIter.__del__ at 0x7ceaeb296c00>

Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
self._shutdown_workers()    if w.is_alive():
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers

      File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
if w.is_alive():
Exception ignored in:     <function _MultiProcessingDataLoaderIter.__del__ at 0x7ceaeb296c00>assert self._parent_pid == os.getpid(), 'can o

val:   0%|          | 0/11 [00:00<?, ?it/s]

epoch  30 | train loss 1.0387 | val loss 1.2609 disease=0.8882 | 220s
[train] best val disease acc 0.8912 @ epoch 25
[train] saved: /content/drive/MyDrive/Assignment2COS30082/outputs/indep_disease_effb0/best.pt
[train] saved: /content/drive/MyDrive/Assignment2COS30082/outputs/indep_disease_effb0/best_weights_only.pt
Disease checkpoint: /content/drive/MyDrive/Assignment2COS30082/outputs/indep_disease_effb0/best.pt
Files: ['config.yaml', 'history.csv', 'last.pt', 'best.pt', 'best_weights_only.pt']


In [15]:
# Verify both checkpoints exist and are loadable
for path in (CROP_CKPT, DISEASE_CKPT):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    print(f"\n{path}")
    print(f"  keys: {list(ckpt.keys())}")
    print(f"  heads: {ckpt['heads']}")
    print(f"  epoch: {ckpt['epoch']}")
    print(f"  config.mode: {ckpt['config'].get('mode')}")
    print(f"  config.data.root: {ckpt['config']['data']['root']}")
    print(f"  state_dict tensors: {len(ckpt['model'])}")



/content/drive/MyDrive/Assignment2COS30082/outputs/indep_crop_effb0/best.pt
  keys: ['model', 'heads', 'config', 'epoch', 'val', 'num_crops', 'num_diseases']
  heads: {'crop': 55}
  epoch: 19
  config.mode: crop
  config.data.root: /content/drive/MyDrive/Assignment2COS30082
  state_dict tensors: 360

/content/drive/MyDrive/Assignment2COS30082/outputs/indep_disease_effb0/best.pt
  keys: ['model', 'heads', 'config', 'epoch', 'val', 'num_crops', 'num_diseases']
  heads: {'disease': 175}
  epoch: 25
  config.mode: disease
  config.data.root: /content/drive/MyDrive/Assignment2COS30082
  state_dict tensors: 360


In [16]:
# Plot training curves for both independent models
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

crop_hist = Path(OUTPUT_DIR) / "indep_crop_effb0" / "history.csv"
disease_hist = Path(OUTPUT_DIR) / "indep_disease_effb0" / "history.csv"

df_crop = pd.read_csv(crop_hist)
df_disease = pd.read_csv(disease_hist)

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)

# ---- Crop model ----
ax = axes[0, 0]
ax.plot(df_crop["epoch"], df_crop["train_loss"], label="train")
ax.plot(df_crop["epoch"], df_crop["val_loss"], label="val")
ax.set_title("Crop model – Loss")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(df_crop["epoch"], df_crop["train_crop_acc"], label="train")
ax.plot(df_crop["epoch"], df_crop["val_crop_acc"], label="val")
ax.set_title("Crop model – Accuracy")
ax.set_ylabel("Accuracy")
ax.legend()
ax.grid(True, alpha=0.3)

# ---- Disease model ----
ax = axes[1, 0]
ax.plot(df_disease["epoch"], df_disease["train_loss"], label="train")
ax.plot(df_disease["epoch"], df_disease["val_loss"], label="val")
ax.set_title("Disease model – Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(df_disease["epoch"], df_disease["train_disease_acc"], label="train")
ax.plot(df_disease["epoch"], df_disease["val_disease_acc"], label="val")
ax.set_title("Disease model – Accuracy")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Optional: also save the figure
save_path = Path(OUTPUT_DIR) / "indep_effb0_curves" / "training_curves.png"
save_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Saved: {save_path}")

Saved: /content/drive/MyDrive/Assignment2COS30082/outputs/indep_effb0_curves/training_curves.png


## 6. Metrics

In [17]:
def log_softmax(x: np.ndarray) -> np.ndarray:
    x = x - x.max(axis=1, keepdims=True)
    return x - np.log(np.exp(x).sum(axis=1, keepdims=True))


def topk_accuracy(scores: np.ndarray, targets: np.ndarray, ks=(1, 5)) -> dict[str, float]:
    if len(targets) == 0:
        return {f"top{k}": float("nan") for k in ks}
    order = np.argsort(-scores, axis=1)
    out = {}
    for k in ks:
        kk = min(k, scores.shape[1])
        out[f"top{k}"] = float((order[:, :kk] == targets[:, None]).any(axis=1).mean())
    return out


def composition_scores(crop_logits, disease_logits, pairs):
    p = np.asarray(pairs)
    return log_softmax(crop_logits)[:, p[:, 0]] + log_softmax(disease_logits)[:, p[:, 1]]


def add_unseen_bias(scores, num_seen, bias):
    s = scores.copy()
    s[:, num_seen:] += bias
    return s


def harmonic_mean(a, b):
    return 0.0 if a + b == 0 else 2 * a * b / (a + b)


def _trapezoid(y, x):
    fn = getattr(np, "trapezoid", None) or np.trapz
    return float(fn(y, x))


def calibration_sweep(seen_scores, seen_gt, unseen_scores, unseen_gt, num_seen, n_points=201):
    allsc = np.concatenate([seen_scores, unseen_scores])
    gaps = allsc[:, :num_seen].max(1) - allsc[:, num_seen:].max(1)
    biases = np.unique(np.concatenate([
        np.quantile(gaps, np.linspace(0, 1, n_points)),
        [0.0, gaps.min() - 1e-3, gaps.max() + 1e-3],
    ]))
    seen_acc, unseen_acc = [], []
    for b in biases:
        seen_acc.append(float((add_unseen_bias(seen_scores, num_seen, b).argmax(1) == seen_gt).mean()))
        unseen_acc.append(float((add_unseen_bias(unseen_scores, num_seen, b).argmax(1) == unseen_gt).mean()))
    seen_acc, unseen_acc = np.array(seen_acc), np.array(unseen_acc)
    hms = np.array([harmonic_mean(s, u) for s, u in zip(seen_acc, unseen_acc)])
    best = int(hms.argmax())
    order = np.argsort(seen_acc)
    return {
        "biases": biases, "seen_acc": seen_acc, "unseen_acc": unseen_acc, "hm": hms,
        "best_hm": float(hms[best]), "best_bias": float(biases[best]),
        "seen_at_best": float(seen_acc[best]), "unseen_at_best": float(unseen_acc[best]),
        "auc": _trapezoid(unseen_acc[order], seen_acc[order]),
    }

print("metrics ready")


metrics ready


## 7. Evaluate (load both checkpoints)

In [18]:
def load_independent_models(crop_path, disease_path, device):
    """Load crop + disease checkpoints. Rebuilds models from embedded config."""
    models, cfg, seen_heads = [], None, set()
    for path in (crop_path, disease_path):
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        assert "model" in ckpt and "heads" in ckpt and "config" in ckpt, \
            f"Checkpoint missing keys: {list(ckpt.keys())}"
        clash = seen_heads & set(ckpt["heads"])
        if clash:
            raise ValueError(f"head clash {clash}")
        seen_heads |= set(ckpt["heads"])
        model = build_model(ckpt["config"], ckpt["heads"], pretrained=False)
        model.load_state_dict(ckpt["model"])
        models.append(model.to(device).eval())
        cfg = cfg or ckpt["config"]
        print(f"Loaded {path}: heads={ckpt['heads']} epoch={ckpt['epoch']}")
    if seen_heads != {"crop", "disease"}:
        raise ValueError(f"need crop + disease heads, got {seen_heads}")
    return models, cfg


@torch.no_grad()
def collect_logits(models, loader, device):
    out = {}
    for batch in tqdm(loader, leave=False):
        x = batch["image"].to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            for m in models:
                for k, v in m(x).items():
                    out.setdefault(k, []).append(v.float().cpu())
    return {k: torch.cat(v).numpy() for k, v in out.items()}


def gt_pair_indices(df, ls):
    idx = ls.pair_index()
    return np.array([idx.get(p, -1) for p in zip(df["crop_label"].tolist(), df["disease_label"].tolist())])


def save_calibration_plot(cal, out_dir):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(cal["seen_acc"], cal["unseen_acc"], lw=1.5)
    ax.scatter([cal["seen_at_best"]], [cal["unseen_at_best"]], color="red", zorder=3,
               label=f"best HM {cal['best_hm']:.3f}")
    ax.set_xlabel("seen top-1 accuracy")
    ax.set_ylabel("unseen top-1 accuracy")
    ax.set_title(f"Seen-unseen trade-off (AUC {cal['auc']:.3f})")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "calibration_curve.png"), dpi=150)
    plt.close(fig)
    print("saved calibration_curve.png")


def evaluate_independent(crop_ckpt, disease_ckpt, out, batch_size=128):
    os.makedirs(out, exist_ok=True)
    device = get_device()
    models, cfg = load_independent_models(crop_ckpt, disease_ckpt, device)
    dc = cfg["data"]

    root = dc["root"]
    if not os.path.isdir(root):
        root = DATA_ROOT
        print(f"[warn] ckpt data.root missing, falling back to {root}")

    train_df = read_split(os.path.join(root, dc["train_csv"]))
    seen_df = read_split(os.path.join(root, dc["test_seen_csv"]))
    unseen_df = read_split(os.path.join(root, dc["test_unseen_csv"]))
    ls = build_label_space(train_df, seen_df, unseen_df)
    audit(train_df, seen_df, unseen_df, ls)

    tf = build_transforms(dc["img_size"], train=False)
    image_dir = os.path.join(root, dc["image_dir"])
    results, scores_by_split, gt_by_split, pred_rows = {}, {}, {}, []

    for split, df in (("seen", seen_df), ("unseen", unseen_df)):
        gt = gt_pair_indices(df, ls)
        keep = gt >= 0
        df, gt = df[keep].reset_index(drop=True), gt[keep]
        loader = DataLoader(PlantDiseaseDataset(df, image_dir, tf),
                            batch_size=batch_size, num_workers=dc.get("num_workers", 2),
                            shuffle=False)
        logits = collect_logits(models, loader, device)
        scores = composition_scores(logits["crop"], logits["disease"], ls.pairs)
        scores_by_split[split], gt_by_split[split] = scores, gt

        r = {
            "n_images": int(len(df)),
            "crop": topk_accuracy(logits["crop"], df["crop_label"].to_numpy()),
            "disease": topk_accuracy(logits["disease"], df["disease_label"].to_numpy()),
            "composition": topk_accuracy(scores, gt),
        }
        if split == "seen":
            r["composition_closed_world"] = topk_accuracy(scores[:, : ls.num_seen], gt)
        else:
            r["composition_closed_world"] = topk_accuracy(scores[:, ls.num_seen:], gt - ls.num_seen)
        results[split] = r

        top5 = np.argsort(-scores, axis=1)[:, :5]
        for i in range(len(df)):
            pred_rows.append({
                "split": split, "image": df["image"][i], "true": ls.pair_name(gt[i]),
                **{f"pred_{k+1}": ls.pair_name(top5[i, k]) for k in range(5)},
            })

    results["harmonic_mean"] = {
        k: harmonic_mean(results["seen"]["composition"][k], results["unseen"]["composition"][k])
        for k in ("top1", "top5")
    }
    cal = calibration_sweep(scores_by_split["seen"], gt_by_split["seen"],
                            scores_by_split["unseen"], gt_by_split["unseen"], ls.num_seen)
    results["calibrated"] = {k: cal[k] for k in
                             ("best_hm", "best_bias", "seen_at_best", "unseen_at_best", "auc")}
    results["checkpoints"] = [crop_ckpt, disease_ckpt]
    results["method"] = "independent_cnn"

    with open(os.path.join(out, "metrics.json"), "w") as f:
        json.dump(results, f, indent=2)
    pd.DataFrame(pred_rows).to_csv(os.path.join(out, "predictions.csv"), index=False)

    pred = scores_by_split["seen"].argmax(1)
    n_pairs = len(ls.pairs)
    cm = confusion_matrix(gt_by_split["seen"], pred, labels=np.arange(n_pairs))[: ls.num_seen]
    np.save(os.path.join(out, "confusion_seen_compositions.npy"), cm)
    off = cm.copy()
    np.fill_diagonal(off[:, : ls.num_seen], 0)
    rows, cols = np.nonzero(off)
    top = sorted(zip(rows, cols, off[rows, cols]), key=lambda t: -t[2])[:30]
    pd.DataFrame([{"true": ls.pair_name(r), "predicted": ls.pair_name(c), "count": int(n),
                   "rate_of_true_class": n / max(cm[r].sum(), 1)} for r, c, n in top]
                 ).to_csv(os.path.join(out, "top_confusions_seen.csv"), index=False)

    save_calibration_plot(cal, out)

    print("\n                       Top-1    Top-5")
    for split in ("seen", "unseen"):
        for key in ("crop", "disease", "composition", "composition_closed_world"):
            r = results[split][key]
            print(f"{split:>6} {key:<25} {r['top1']:.4f}   {r['top5']:.4f}")
    hm = results["harmonic_mean"]
    print(f"harmonic mean (composition)      {hm['top1']:.4f}   {hm['top5']:.4f}")
    c = results["calibrated"]
    print(f"calibrated best HM {c['best_hm']:.4f} "
          f"(seen {c['seen_at_best']:.4f}, unseen {c['unseen_at_best']:.4f}, "
          f"bias {c['best_bias']:.3f}) | AUC {c['auc']:.4f}")
    print(f"\nsaved to {out}")
    return results, cm, ls

print("evaluate_independent() ready")


evaluate_independent() ready


In [19]:
EVAL_OUT = os.path.join(OUTPUT_DIR, "indep_effb0", "eval")
RESULTS, CM, LS = evaluate_independent(CROP_CKPT, DISEASE_CKPT, EVAL_OUT)


Loaded /content/drive/MyDrive/Assignment2COS30082/outputs/indep_crop_effb0/best.pt: heads={'crop': 55} epoch=19
Loaded /content/drive/MyDrive/Assignment2COS30082/outputs/indep_disease_effb0/best.pt: heads={'disease': 175} epoch=25
[data] train=8000 test_seen=2000 test_unseen=1448
[data] crops=55 diseases=175 seen pairs=323 unseen pairs=10
[data] compositional zero-shot setting
[data] images/seen-pair: min=1 median=7 max=544


  0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

saved calibration_curve.png

                       Top-1    Top-5
  seen crop                      0.9010   0.9600
  seen disease                   0.6890   0.8855
  seen composition               0.6840   0.8860
  seen composition_closed_world  0.6845   0.8865
unseen crop                      0.7210   0.8439
unseen disease                   0.0988   0.2859
unseen composition               0.0849   0.3543
unseen composition_closed_world  0.7728   0.9544
harmonic mean (composition)      0.1511   0.5062
calibrated best HM 0.6109 (seen 0.5900, unseen 0.6333, bias 6.433) | AUC 0.4877

saved to /content/drive/MyDrive/Assignment2COS30082/outputs/indep_effb0/eval


## 8. Confusion matrix plots

In [20]:
SURFACE, INK, INK_2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
WRONG_DISEASE, WRONG_CROP = "#2a78d6", "#eb6834"
RAMP = LinearSegmentedColormap.from_list(
    "blue", ["#cde2fb", "#86b6ef", "#3987e5", "#2a78d6", "#1c5cab", "#104281", "#0d366b"])
RAMP.set_bad(SURFACE)

STYLE = {
    "font.family": ["DejaVu Sans", "sans-serif"],
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK_2, "axes.edgecolor": GRID,
    "xtick.color": INK_2, "ytick.color": INK_2, "axes.titleweight": "bold",
}


def crop_level(cm, ls):
    pairs = np.asarray(ls.pairs)
    rows = np.eye(ls.num_crops, dtype=int)[pairs[: cm.shape[0], 0]]
    cols = np.eye(ls.num_crops, dtype=int)[pairs[: cm.shape[1], 0]]
    return rows.T @ cm @ cols


def _heatmap(ax, counts, row_names, col_names, show_counts=False, font=6.5):
    support = counts.sum(1, keepdims=True)
    share = counts / np.clip(support, 1, None)
    im = ax.imshow(np.ma.masked_equal(share, 0), cmap=RAMP, vmin=0, vmax=1,
                   interpolation="nearest", aspect="auto")
    for r, c in zip(*np.nonzero(counts)):
        pct = share[r, c] * 100
        text = f"{pct:.0f}" if pct >= 1 else "<1"
        if show_counts:
            text = f"{text}%\n{counts[r, c]}"
        ax.text(c, r, text, ha="center", va="center", fontsize=font,
                color="white" if share[r, c] > 0.45 else INK)
    ax.set_xticks(range(len(col_names)), col_names, rotation=90)
    ax.set_yticks(range(len(row_names)),
                  [f"{n}  ({int(s)})" for n, s in zip(row_names, support[:, 0])])
    ax.set_xticks(np.arange(-0.5, len(col_names)), minor=True)
    ax.set_yticks(np.arange(-0.5, len(row_names)), minor=True)
    ax.grid(which="minor", color=GRID, linewidth=0.5)
    ax.tick_params(which="minor", length=0)
    ax.tick_params(which="major", length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    return im


def plot_crop_matrix(ccm, names, path):
    keep = np.nonzero(ccm.sum(1) + ccm.sum(0))[0]
    ccm, names = ccm[np.ix_(keep, keep)], [names[i] for i in keep]
    n = len(names)
    acc = np.trace(ccm) / max(ccm.sum(), 1)
    with plt.rc_context({**STYLE, "xtick.labelsize": 7, "ytick.labelsize": 7}):
        fig, ax = plt.subplots(figsize=(0.3 * n + 3, 0.3 * n + 1.5))
        im = _heatmap(ax, ccm, names, names)
        ax.set_xlabel("predicted crop")
        ax.set_ylabel("true crop  (number of test images)")
        ax.set_title(f"Crop confusion (seen test): accuracy {acc:.1%}\n"
                     "cells = % of each true crop; blank = 0", loc="left")
        cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01)
        cbar.outline.set_visible(False)
        cbar.set_label("share of true crop images")
        fig.tight_layout()
        fig.savefig(path, dpi=150)
        plt.close(fig)


def plot_worst_crops(ccm, names, path, k=15, min_images=5):
    support = ccm.sum(1)
    recall = np.where(support >= min_images, np.diag(ccm) / np.clip(support, 1, None), np.inf)
    worst = [i for i in np.argsort(recall, kind="stable") if recall[i] < 1][:k]
    if not worst:
        return
    targets = sorted({c for r in worst for c in np.nonzero(ccm[r])[0]} - set(worst),
                     key=lambda c: names[c])
    cols = worst + targets
    sub = ccm[np.ix_(worst, cols)]
    with plt.rc_context({**STYLE, "xtick.labelsize": 9, "ytick.labelsize": 9}):
        fig, ax = plt.subplots(figsize=(0.55 * len(cols) + 3.5, 0.55 * len(worst) + 2.5))
        _heatmap(ax, sub, [names[i] for i in worst], [names[i] for i in cols],
                 show_counts=True, font=7)
        ax.axvline(len(worst) - 0.5, color=MUTED, linewidth=1)
        ax.set_xlabel("predicted crop")
        ax.set_ylabel("true crop")
        ax.set_title(f"Worst {len(worst)} crops (seen test, >={min_images} images)", loc="left")
        fig.tight_layout()
        fig.savefig(path, dpi=150)
        plt.close(fig)


def plot_top_confusions(cm, ls, path, k=20):
    off = cm.copy()
    np.fill_diagonal(off[:, : cm.shape[0]], 0)
    rows, cols = np.nonzero(off)
    order = np.argsort(-off[rows, cols], kind="stable")[:k]
    rows, cols = rows[order], cols[order]
    if not len(rows):
        return
    pairs = np.asarray(ls.pairs)
    counts = off[rows, cols]
    support = cm[rows].sum(1)
    same_crop = pairs[rows, 0] == pairs[cols, 0]
    labels = [f"{ls.pair_name(r)}  ->  {ls.disease_names.get(pairs[c, 1], pairs[c, 1])}" if s
              else f"{ls.pair_name(r)}  ->  {ls.pair_name(c)}"
              for r, c, s in zip(rows, cols, same_crop)]

    with plt.rc_context({**STYLE, "ytick.labelsize": 9}):
        fig, ax = plt.subplots(figsize=(11, 0.36 * len(rows) + 1.8))
        y = np.arange(len(rows))
        ax.barh(y, counts, height=0.7, color=np.where(same_crop, WRONG_DISEASE, WRONG_CROP),
                edgecolor=SURFACE, linewidth=1)
        for yi, n, s in zip(y, counts, support):
            ax.text(n + counts.max() * 0.01, yi, f"{n} of {s}  ({n / s:.0%})",
                    va="center", fontsize=8.5, color=INK_2)
        ax.set_yticks(y, labels)
        ax.invert_yaxis()
        ax.set_xlim(0, counts.max() * 1.18)
        ax.set_xlabel("misclassified test images")
        ax.xaxis.grid(True, color=GRID, linewidth=0.6)
        ax.set_axisbelow(True)
        ax.tick_params(length=0)
        for side in ("top", "right", "left"):
            ax.spines[side].set_visible(False)
        ax.set_title(f"Top {len(rows)} mix-ups (seen test)", loc="left")
        handles = [Patch(color=WRONG_DISEASE, label=f"right crop, wrong disease ({same_crop.sum()})"),
                   Patch(color=WRONG_CROP, label=f"wrong crop ({(~same_crop).sum()})")]
        ax.legend(handles=[h for h, n in zip(handles, (same_crop.sum(), (~same_crop).sum())) if n],
                  loc="lower right", frameon=False)
        fig.tight_layout()
        fig.savefig(path, dpi=150)
        plt.close(fig)


def plot_all(cm, ls, out_dir):
    ccm = crop_level(cm, ls)
    names = [ls.crop_names.get(i, str(i)) for i in range(ls.num_crops)]
    plot_crop_matrix(ccm, names, os.path.join(out_dir, "confusion_seen_crops.png"))
    plot_worst_crops(ccm, names, os.path.join(out_dir, "confusion_seen_crops_worst.png"))
    plot_top_confusions(cm, ls, os.path.join(out_dir, "top_confusions_seen.png"))
    print(f"plots saved to {out_dir}")

plot_all(CM, LS, EVAL_OUT)


plots saved to /content/drive/MyDrive/Assignment2COS30082/outputs/indep_effb0/eval


## 9. What was saved on Drive

```
DRIVE_ROOT/outputs/
  indep_crop_effb0/
    best.pt                 # full checkpoint (weights + heads + config)  ← use this
    best_weights_only.pt    # state_dict only
    last.pt
    config.yaml
    history.csv
  indep_disease_effb0/
    best.pt
    best_weights_only.pt
    last.pt
    config.yaml
    history.csv
  indep_effb0/eval/
    metrics.json
    predictions.csv
    confusion_seen_compositions.npy
    top_confusions_seen.csv
    calibration_curve.png
    confusion_seen_crops.png
    confusion_seen_crops_worst.png
    top_confusions_seen.png
```

**Why checkpoints work:** each `best.pt` stores `model`, `heads`, and `config`.  
Evaluation rebuilds the architecture from `ckpt["config"]` + `ckpt["heads"]`, then loads `ckpt["model"]`. Train and eval stay in sync even if you restart the runtime later.


In [21]:
print("Crop dir:", CROP_DIR)
print(" ", os.listdir(CROP_DIR))
print("Disease dir:", DISEASE_DIR)
print(" ", os.listdir(DISEASE_DIR))
print("Eval dir:", EVAL_OUT)
print(" ", os.listdir(EVAL_OUT))
print("\nmetrics.json harmonic_mean:", RESULTS["harmonic_mean"])
print("calibrated:", RESULTS["calibrated"])


Crop dir: /content/drive/MyDrive/Assignment2COS30082/outputs/indep_crop_effb0
  ['config.yaml', 'history.csv', 'last.pt', 'best.pt', 'best_weights_only.pt']
Disease dir: /content/drive/MyDrive/Assignment2COS30082/outputs/indep_disease_effb0
  ['config.yaml', 'history.csv', 'last.pt', 'best.pt', 'best_weights_only.pt']
Eval dir: /content/drive/MyDrive/Assignment2COS30082/outputs/indep_effb0/eval
  ['metrics.json', 'predictions.csv', 'confusion_seen_compositions.npy', 'top_confusions_seen.csv', 'calibration_curve.png', 'confusion_seen_crops.png', 'confusion_seen_crops_worst.png', 'top_confusions_seen.png']

metrics.json harmonic_mean: {'top1': 0.15112193649904082, 'top5': 0.5061650578419625}
calibrated: {'best_hm': 0.610877763475826, 'best_bias': 6.43303729057312, 'seen_at_best': 0.59, 'unseen_at_best': 0.6332872928176796, 'auc': 0.4877209944751381}
